In [ ]:
!pip install llama-index-llms-cohere pandas openpyxl python-dotenv

import os
import json
import pandas as pd
from dotenv import load_dotenv
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage

load_dotenv()
cohere_key = os.getenv("COHERE_API_KEY")

if not cohere_key:
    print("Error: COHERE_API_KEY is missing!")
else:
    llm = Cohere(model="command-r-08-2024", api_key=cohere_key) 
    print("Cohere LLM is ready with the latest stable model!")

def ask_ai(prompt):
    try:
        messages = [ChatMessage(role="user", content=prompt)]
        response = llm.chat(messages)
        return response.message.content.strip()
    except Exception as e:
        print(f"API Error: {e}")
        return "ERROR"

In [ ]:
# ---  סיווג עם הנחיות למניעת ניחושים ---

VALID_CATEGORIES = ["רכב", "דירה", "בריאות", "מידע כללי"]

with open("questions.json", "r", encoding="utf-8") as f:
    original_questions = json.load(f)

categorized_dict = {cat: [] for cat in VALID_CATEGORIES}

print("מתחיל לסווג שאלות עם לוגיקה משופרת...")

for q in original_questions:
    # פרומפט שמגדיר מתי להשתמש ב-General
    prompt_cat = f"""Classify this insurance question into one of these categories: 'Car', 'Home', 'Health', or 'General'.
    
    Rules:
    1. If the question mentions cars, vehicles, or driving, answer 'Car'.
    2. If the question mentions houses, apartments, or property, answer 'Home'.
    3. If the question mentions medical, surgery, or health, answer 'Health'.
    4. If the question is about customer service, general company recommendations, or speed of service WITHOUT mentioning a specific insurance type, you MUST answer 'General'.
    
    Answer only with the category name.
    Question: {q}"""
    
    raw_response = ask_ai(prompt_cat)
    cat_response = raw_response.lower().strip()
    
    print(f"שאלה: {q[:30]}... | תשובת AI: '{cat_response}'")

    # בדיקת מילות מפתח בתוך התשובה
    if any(w in cat_response for w in ["car", "vehicle", "רכב"]):
        target_cat = "רכב"
    elif any(w in cat_response for w in ["home", "house", "apartment", "דירה"]):
        target_cat = "דירה"
    elif any(w in cat_response for w in ["health", "medical", "בריאות"]):
        target_cat = "בריאות"
    else:
        # זה יתפוס את 'general' וגם מקרים של שגיאה
        target_cat = "מידע כללי"
    
    categorized_dict[target_cat].append({"שאלה": q, "מקור": "Original"})

# ניקוי קטגוריות ריקות
categorized_dict = {k: v for k, v in categorized_dict.items() if len(v) > 0}

print("\n--- סיכום סיווג מעודכן ---")
for cat, qs in categorized_dict.items():
    print(f"קטגוריה: {cat} | מספר שאלות: {len(qs)}")

    

In [ ]:
# ---  : שאלות ייעוץ והשוואה לפי קטגוריה ---

for cat in list(categorized_dict.keys()):
    print(f"מייצר שאלות ייעוץ ספציפיות עבור: {cat}...")
    
    # פרומפט שדורש התלבטות צרכנית שקשורה ישירות לקטגוריה
    prompt_gen = f"""You are an Israeli consumer looking for {cat} insurance. 
    You are consulting an AI to help you CHOOSE the best company and policy for {cat} specifically.
    
    Write 5 natural questions that a person asks when they are debating between companies.
    
    Examples of the style needed:
    - For 'רכב': Ask about service in claims, or which company is best for young drivers.
    - For 'דירה': Ask about water damage service or if a certain company is reliable for earthquakes.
    - For 'בריאות': Ask about comparing covers for surgeries or which company has the fastest approvals.
    
    Rules:
    1. The questions MUST be about {cat} insurance specifically.
    2. Focus on: Comparison, reliability, hidden 'small print', and price vs. value.
    3. Write in HEBREW. 
    4. Return ONLY the questions, one per line, no numbers, no intro."""
    
    new_qs_text = ask_ai(prompt_gen)
    
    for nq in new_qs_text.split('\n'):
        clean_q = nq.strip().lstrip('0123456789.- ')
        
        # סינון נוסף לוודא שה-AI לא סטה מהנושא
        if len(clean_q) > 15 and clean_q.lower() != "error":
            categorized_dict[cat].append({"שאלה": clean_q, "מקור": "AI Generated (Consultation)"})

print("\nהסוכן סיים לייצר את שאלות הייעוץ הממוקדות!")

In [ ]:
# ---  קבלת תשובות מהמודל עבור כל שאלה ---

print("מתחיל לקבל תשובות מהמודל עבור כל השאלות (זה עשוי לקחת זמן)...")

for cat in categorized_dict:
    print(f"מעבד תשובות עבור קטגוריית: {cat}")
    for item in categorized_dict[cat]:
        question = item["שאלה"]
        
        # כאן אנחנו מבקשים מה-AI לענות על השאלה כיועץ ביטוח
        prompt_ans = f"אתה יועץ ביטוח מומחה. ענה בקצרה וביעילות על השאלה הבאה עבור לקוח מתלבט: {question}"
        
        answer = ask_ai(prompt_ans)
        
        # שמירת התשובה בתוך המילון של השאלה
        item["תשובת Cohere"] = answer

print("\nסיימתי לאסוף את כל התשובות!")

מתחיל לקבל תשובות מהמודל עבור כל השאלות (זה עשוי לקחת זמן)...
מעבד תשובות עבור קטגוריית: רכב
מעבד תשובות עבור קטגוריית: דירה
מעבד תשובות עבור קטגוריית: בריאות
מעבד תשובות עבור קטגוריית: מידע כללי

סיימתי לאסוף את כל התשובות!


In [ ]:
# --- ניתוח מחקרי של התשובות ---

print("מתחיל בניתוח מחקרי של התשובות עבור ביטוח ישיר...")

for cat in categorized_dict:
    print(f"מנתח נתונים בקטגוריית: {cat}")
    for item in categorized_dict[cat]:
        # לוקחים את השאלה והתשובה המקורית שקיבלנו מה-AI
        question = item["שאלה"]
        ai_answer = item.get("תשובת Cohere", "")
        
        if ai_answer == "ERROR" or not ai_answer:
            continue

        # פרומפט הניתוח המחקרי
        analysis_prompt = f"""
        נתח את התשובה הבאה שניתנה על ידי מודל שפה לגבי ביטוח {cat}.
        השאלה הייתה: {question}
        התשובה שניתנה: {ai_answer}
        
        עליך להחזיר תשובה בפורמט JSON בלבד עם המפתחות הבאים:
        1. "חוזקות": אילו חוזקות של חברות ביטוח צוינו בתשובה?
        2. "חולשות": אילו חולשות או חסרונות צוינו בתשובה?
        3. "מצב_ביטוח_ישיר": רמת ההמלצה על 'ביטוח ישיר' (גבוהה/בינונית/נמוכה/לא צוין).
        4. "נימוק_המצב": מדוע המודל המליץ כך? מה הוא רואה כיתרון או חיסרון בביטוח ישיר?
        5. "מסקנה_אופרטיבית": מה ביטוח ישיר צריכה לעשות כדי שביצועיה והמלצת המודל עליה ישתפרו בשאלה זו?
        
        החזר רק JSON תקין בעברית.
        """
        
        # קבלת הניתוח (נשתמש ב-ask_ai)
        analysis_raw = ask_ai(analysis_prompt)
        
        try:
            # ניסיון לחלץ את הנתונים מה-JSON שה-AI החזיר
            # לעיתים ה-AI מוסיף טקסט מיותר לפני ה-JSON, לכן ננקה אותו
            start_idx = analysis_raw.find('{')
            end_idx = analysis_raw.rfind('}') + 1
            analysis_json = json.loads(analysis_raw[start_idx:end_idx])
            
            item["חוזקות חברות"] = analysis_json.get("חוזקות", "")
            item["חולשות חברות"] = analysis_json.get("חולשות", "")
            item["מצב ביטוח ישיר"] = analysis_json.get("מצב_ביטוח_ישיר", "")
            item["נימוק המצב"] = analysis_json.get("נימוק_המצב", "")
            item["מסקנה לקידום"] = analysis_json.get("מסקנה_אופרטיבית", "")
            
        except:
            # אם ה-AI לא החזיר JSON תקין, נשמור את הטקסט הגולמי בעמודה אחת
            item["ניתוח כללי"] = analysis_raw

        # גם כאן כדאי להוסיף השהיה קטנה כדי לא להיחסם
        # time.sleep(10)

print("\nהניתוח המחקרי הסתיים!")

מתחיל בניתוח מחקרי של התשובות עבור ביטוח ישיר...
מנתח נתונים בקטגוריית: רכב
מנתח נתונים בקטגוריית: דירה
מנתח נתונים בקטגוריית: בריאות
מנתח נתונים בקטגוריית: מידע כללי

הניתוח המחקרי הסתיים!


In [ ]:
# filename = "organized_insurance_audit.xlsx"

# with pd.ExcelWriter(filename, engine='openpyxl') as writer:
#     for cat, data in categorized_dict.items():
#         # ניקוי שם הגליון (מקסימום 31 תווים)
#         sheet_name = cat[:31]
        
#         df = pd.DataFrame(data)
#         df.to_excel(writer, sheet_name=sheet_name, index=False)
#         print(f"הגליון '{sheet_name}' נשמר עם {len(df)} שאלות.")

# print(f"\nבוצע בהצלחה! הקובץ נמצא כאן: {os.path.abspath(filename)}")

# --- תא 5: יצירת האקסל המושקע ---
# import os

# filename = "insurance_expert_answers.xlsx"

# # אנחנו בונים את הקובץ
# with pd.ExcelWriter(filename, engine='openpyxl') as writer:
#     for cat, data in categorized_dict.items():
#         df = pd.DataFrame(data)
        
#         # אנחנו מוודאים שהעמודות מסודרות נכון באקסל
#         # עמודה 1: השאלה | עמודה 2: מקור | עמודה 3: התשובה של המודל
#         if "תשובת Cohere" in df.columns:
#             cols = ["שאלה", "מקור", "תשובת Cohere"]
#             df = df[cols]
        
#         df.to_excel(writer, sheet_name=cat[:31], index=False)
#         print(f"הגליון '{cat}' נוצר בהצלחה.")

# print(f"\nהקובץ המלא מוכן! הנתיב שלו:")
# print(os.path.abspath(filename))

# --- תא 6: שמירת דוח המחקר הסופי ---

filename = "Direct_Insurance_Research_Report.xlsx"

with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    for cat, data in categorized_dict.items():
        df = pd.DataFrame(data)
        
        # הגדרת סדר העמודות החדש
        final_cols = [
            "שאלה", "מקור", "תשובת Cohere", 
            "חוזקות חברות", "חולשות חברות", 
            "מצב ביטוח ישיר", "נימוק המצב", "מסקנה לקידום"
        ]
        
        # מוודא שכל העמודות קיימות ב-DF (כדי למנוע שגיאה)
        existing_cols = [c for c in final_cols if c in df.columns]
        df = df[existing_cols]
        
        df.to_excel(writer, sheet_name=cat[:31], index=False)

print(f"המחקר הסתיים בהצלחה! הקובץ נוצר: {filename}")

המחקר הסתיים בהצלחה! הקובץ נוצר: Direct_Insurance_Research_Report.xlsx
